In [1]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from tarnet import Tarnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/RERUM")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
sys.path.append("..")
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [2]:
%time train_df = pd.read_csv(r"../dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"../dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"../dataset/Hillstrom/Men/val_men.csv")

CPU times: user 25.3 ms, sys: 9.16 ms, total: 34.5 ms
Wall time: 33.9 ms
CPU times: user 11.1 ms, sys: 6.01 ms, total: 17.1 ms
Wall time: 17.1 ms
CPU times: user 3.5 ms, sys: 2.98 ms, total: 6.48 ms
Wall time: 6.48 ms


In [3]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [4]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [5]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [6]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [7]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [8]:
epochs = 150
early_stop_metric = "ema_qini"
ema = True
ema_alpha = 0.25
patience = 20
early_stop_start = 30
print (f" epochs = {epochs}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" early stop start = {early_stop_start}")

 epochs = 150
 early stop = ema_qini
 use ema = True
 ema alpha = 0.25
 patience = 20
 early stop start = 30


In [9]:
import io
import optuna
from contextlib import redirect_stdout, redirect_stderr

# Minimize Optuna console noise
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Optuna search config (validation only)
seeds = [412312, 42, 1874, 902745, 1]
n_trials = 50
tpe_sampler_seed = 412312
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    grid_lr = trial.suggest_float("lr", 1e-5,  1e-3, log=True)
    grid_wd = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    grid_outcome_dropout = trial.suggest_float("outcome_dropout", 0.0, 0.5)
    grid_shared_dropout = trial.suggest_float("shared_dropout", 0.0, 0.5)
    grid_outcome_hidden = trial.suggest_int("outcome_hidden", 50, 400, step=50)
    grid_shared_hidden = trial.suggest_int("shared_hidden", 50, 400, step=50)
    grid_uplift_ranking = trial.suggest_float("uplift_ranking", 1e-4, 100.0)
    # grid_reponse_ranking = trial.suggest_float("response_ranking", 1e-4, 100.0)
    # grid_ranking_start = trial.suggest_int("ranking_start", 0, 20, step=1)

    val_auqc_list = []
    val_ate_err_list = []
    for SEED in seeds:
        seed_everything(SEED)

        tarnet = Tarnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=grid_shared_hidden,
            outcome_hidden=grid_outcome_hidden,
            outcome_dropout=grid_outcome_dropout,
            shared_dropout=grid_shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            uplift_ranking=grid_uplift_ranking,
            response_ranking=0,
            ranking_start_epoch=15
        )

        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            tarnet.fit(train_loader, val_loader)

        x_men_val_t_on_device = x_men_val_t.to(device)
        y0_pred, y1_pred = tarnet.predict(x_men_val_t_on_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_val_t.detach().cpu().numpy().flatten()
        t_true = t_men_val_t.detach().cpu().numpy().flatten()
        
        current_val_auqc = auqc(y_true, t_true, uplift_pred, bins=100, plot=False)
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
        current_val_ate_err = abs(ate_pred - ate_true)

        val_auqc_list.append(current_val_auqc)
        val_ate_err_list.append(current_val_ate_err)

    # Calculate aggregated metrics across the 5 validation seeds
    mean_auqc = float(np.mean(val_auqc_list))
    std_auqc = float(np.std(val_auqc_list))
    mean_ate_err = float(np.mean(val_ate_err_list))

    # Apply penalty for instability and miscalibration
    penalty_std = std_auqc * 1.0
    penalty_ate = mean_ate_err * 0.05

    final_score = mean_auqc - penalty_std - penalty_ate

    trial.set_user_attr("mean_val_auqc", mean_auqc)
    trial.set_user_attr("std_Val_auqc", std_auqc)
    trial.set_user_attr("mean_val_ate_err", mean_ate_err)
    trial.set_user_attr("final_score", final_score)
    return final_score

def print_trial_callback(study, trial):
    value = float(trial.value) if trial.value is not None else float("nan")
    best_trial = study.best_trial
    best_value = float(best_trial.value) if best_trial.value is not None else float("nan")
    print(
        f"Finished trial {trial.number}: val loss: {value:.4f} - "
        f"with hyperparameters: {trial.params} | "
        f"best trial: {best_trial.number} loss: {best_value:.4f}",
        flush=True
    )

sampler = optuna.samplers.TPESampler(seed=tpe_sampler_seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=n_trials, show_progress_bar=True, callbacks=[print_trial_callback])

trial_rows = []
for t in study.trials:
    if t.value is None:
        continue
    trial_rows.append({
        "trial": t.number,
        "lr": round(float(t.params["lr"]), 4),
        "weight_decay": round(float(t.params["weight_decay"]), 4),
        "shared_hidden": int(t.params["shared_hidden"]),
        "outcome_hidden": int(t.params["outcome_hidden"]),
        "shared_dropout": round(float(t.params["shared_dropout"]), 3),
        "outcome_dropout": round(float(t.params["outcome_dropout"]), 3),
        "uplift_ranking": round(float(t.params["uplift_ranking"]), 4),
        # "response_ranking": round(float(t.params["response_ranking"]), 4),
        # "ranking_start": int(t.params["ranking_start"]),
        "mean_val_auqc": float(t.value),
        "std_Val_auqc": float(t.user_attrs.get("std_Val_auqc", np.nan))
    })

df_grid = pd.DataFrame(trial_rows).sort_values("mean_val_auqc", ascending=True).reset_index(drop=True)
best_params = study.best_params
best_cfg = pd.Series({
    "lr": float(best_params["lr"]),
    "weight_decay": float(best_params["weight_decay"]),
    "shared_hidden": int(best_params["shared_hidden"]),
    "outcome_hidden": int(best_params["outcome_hidden"]),
    "shared_dropout": float(best_params["shared_dropout"]),
    "outcome_dropout": float(best_params["outcome_dropout"]),
    "uplift_ranking": float(best_params["uplift_ranking"]),
    # "response_ranking": float(best_params["response_ranking"]),
    # "ranking_start": int(best_params["ranking_start"]),
    "mean_Val_auqc": float(study.best_value),
    "std_Val_auqc": float(study.best_trial.user_attrs.get("std_Val_auqc", np.nan))
})

/home/ducvu0904/miniconda3/envs/ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/50 [00:00<?, ?it/s]

Finished trial 0: val loss: 0.7293 - with hyperparameters: {'lr': 9.008296332262975e-05, 'weight_decay': 0.00013625464993948335, 'outcome_dropout': 0.2041371296334657, 'shared_dropout': 0.2216925330468017, 'outcome_hidden': 250, 'shared_hidden': 350, 'uplift_ranking': 93.47556399591086} | best trial: 0 loss: 0.7293


Best trial: 0. Best value: 0.729331:   2%|▏         | 1/50 [01:33<1:16:45, 93.99s/it]

Finished trial 1: val loss: 0.7769 - with hyperparameters: {'lr': 0.0009073451879643805, 'weight_decay': 0.000589790629511907, 'outcome_dropout': 0.369412397917029, 'shared_dropout': 0.409458181536563, 'outcome_hidden': 150, 'shared_hidden': 200, 'uplift_ranking': 17.170498906204575} | best trial: 1 loss: 0.7769


Best trial: 1. Best value: 0.776946:   4%|▍         | 2/50 [02:57<1:10:15, 87.82s/it]

...

Best trial: 42. Best value: 0.838613:  96%|█████████▌| 48/50 [1:13:30<02:30, 75.36s/it]

Finished trial 48: val loss: 0.8288 - with hyperparameters: {'lr': 0.0006741260078695676, 'weight_decay': 0.005644539631676795, 'outcome_dropout': 0.1183301961511507, 'shared_dropout': 0.22280842242068646, 'outcome_hidden': 100, 'shared_hidden': 300, 'uplift_ranking': 13.663489674156082} | best trial: 42 loss: 0.8386


Best trial: 42. Best value: 0.838613:  98%|█████████▊| 49/50 [1:14:46<01:15, 75.66s/it]

Finished trial 49: val loss: 0.7971 - with hyperparameters: {'lr': 0.00041378365694889413, 'weight_decay': 0.002417480658795926, 'outcome_dropout': 0.112202574782212, 'shared_dropout': 0.23894554774436494, 'outcome_hidden': 100, 'shared_hidden': 300, 'uplift_ranking': 14.844011073419296} | best trial: 42 loss: 0.8386


Best trial: 42. Best value: 0.838613: 100%|██████████| 50/50 [1:16:01<00:00, 91.23s/it]


In [10]:
from IPython.display import display

if 'study' not in globals():
    print('Run Cell 10 (Optuna tuning) first.')
else:
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best mean_Val_AUQC: {study.best_value:.6f}")
    print(f"Best params: {study.best_params}")

if 'best_cfg' in globals():
    print('\nBest config table:')
    display(best_cfg.to_frame().T)
else:
    print('\nbest_cfg not found.')

if 'df_grid' in globals():
    print('\nTop 10 trials:')
    display(df_grid.head(10))
else:
    print('\ndf_grid not found.')

if 'df_results' in globals():
    print('\nPer-seed test results:')
    display(df_results)
    print('\nTest metrics mean ± std:')
    display(df_results.drop(columns='Seed').agg(['mean', 'std']))

Best trial: 42
Best mean_Val_AUQC: 0.838613
Best params: {'lr': 0.0009666303465207976, 'weight_decay': 0.006912792122967834, 'outcome_dropout': 0.0639204432871469, 'shared_dropout': 0.04926661020063554, 'outcome_hidden': 150, 'shared_hidden': 300, 'uplift_ranking': 58.771307150706704}

Best config table:


,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,uplift_ranking,mean_Val_auqc,std_Val_auqc
0,0.000967,0.006913,300.0,150.0,0.049267,0.06392,58.771307,0.838613,0.008826



Top 10 trials:


,trial,lr,weight_decay,shared_hidden,outcome_hidden,shared_dropout,outcome_dropout,uplift_ranking,mean_val_auqc,std_Val_auqc
0,10,0.0000,0.0100,50,400,0.499,0.020,98.0658,0.349108,0.195771
1,2,0.0000,0.0010,250,250,0.304,0.384,52.2623,0.586915,0.075047
2,18,0.0001,0.0065,100,200,0.176,0.063,87.3281,0.648539,0.036337
3,24,0.0000,0.0012,400,100,0.072,0.002,62.7193,0.658387,0.029072
4,6,0.0001,0.0072,200,200,0.096,0.227,56.6394,0.662597,0.058124
5,14,0.0000,0.0040,150,300,0.011,0.300,69.6744,0.670309,0.075499
6,7,0.0001,0.0001,250,150,0.054,0.258,34.7774,0.695824,0.032029
7,3,0.0002,0.0012,50,150,0.387,0.352,71.2951,0.703718,0.069528
8,0,0.0001,0.0001,350,250,0.222,0.204,93.4756,0.729331,0.025795
9,20,0.0001,0.0019,100,350,0.007,0.293,5.2997,0.749571,0.009581


In [15]:
import pandas as pd
import numpy as np
import torch

# 1. Evaluate selected config on test set (after tuning)
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

if 'best_cfg' not in globals():
    raise ValueError("best_cfg not found. Run grid-search cell first.")

best_lr = float(best_cfg['lr'])
best_wd = float(best_cfg['weight_decay'])
best_shared_hidden = int(best_cfg['shared_hidden'])
best_outcome_hidden = int(best_cfg['outcome_hidden'])
best_shared_dropout = float(best_cfg['shared_dropout'])
best_outcome_dropout = float(best_cfg['outcome_dropout'])
best_uplift_ranking = int(best_cfg['uplift_ranking'])
# best_response_ranking = int(best_cfg['response_ranking'])
# best_ranking_start = int(best_cfg['ranking_start'])

print("Evaluating on test with best validation config:")
print(f"  lr={best_lr:.1e}, weight_decay={best_wd:.1e}")
print(f"  shared_hidden={best_shared_hidden}, outcome_hidden={best_outcome_hidden}")
print(f"  shared_dropout={best_shared_dropout:.3f}, outcome_dropout={best_outcome_dropout:.3f}")
# print(f"  uplift_ranking={best_uplift_ranking}, response_ranking={best_response_ranking}")
print(f"  uplift_ranking={best_uplift_ranking}")
print(f"Number of seeds: {len(seeds)}")

# 2. Loop over seeds for robust test evaluation
for SEED in seeds:
    seed_everything(SEED)

    tarnet = Tarnet(
        input_dim=x_men_train_t.shape[1],
        epochs=epochs,
        learning_rate=best_lr,
        weight_decay=best_wd,
        use_ema=ema,
        ema_alpha=ema_alpha,
        patience=patience,
        shared_hidden=best_shared_hidden,
        outcome_hidden=best_outcome_hidden,
        outcome_dropout=best_outcome_dropout,
        shared_dropout=best_shared_dropout,
        early_stop_metric=early_stop_metric,
        early_stop_start_epoch=early_stop_start,
        uplift_ranking=best_uplift_ranking,
        response_ranking=0,
        ranking_start_epoch=0

    )

    tarnet.fit(train_loader, val_loader)

    # Test prediction
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = tarnet.predict(x_men_test_t_on_device)

    uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
    y_true = y_men_test_t.detach().cpu().numpy().flatten()
    t_true = t_men_test_t.detach().cpu().numpy().flatten()

    # ATE error
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  Done Seed {SEED}")

# 3. Aggregate final test metrics
df_results = pd.DataFrame(all_runs)

print("\n" + "=" * 85)
print(f"{'PER-SEED DETAILS (TEST SET)':^85}")
print("=" * 85)
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format,
    'AUQC': '{:,.4f}'.format,
    'Lift': '{:,.4f}'.format,
    'KRCC': '{:,.4f}'.format,
    'ATE_Err': '{:,.4f}'.format
}))

mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("=" * 85)
print(f"{'TEST SUMMARY (MEAN ± STD)':^85}")
print("-" * 85)
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
print("=" * 85)

Evaluating on test with best validation config:
  lr=9.7e-04, weight_decay=6.9e-03
  shared_hidden=300, outcome_hidden=150
  shared_dropout=0.049, outcome_dropout=0.064
  uplift_ranking=58
Number of seeds: 5
🔃🔃🔃Begin training Tarnet🔃🔃🔃
📊 Early Stop Metric: EMA_QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Best EMA Qini (alpha=0.25)
   Restore to epoch with highest smoothed (EMA) Qini score
   Patience: 20 epochs
Epoch 1/150 | Base Loss: 626.1480 | Uplift Loss: 658.107666 | Total Loss: 1284.2556 | Val Loss: 498.9572 | Val Qini: 0.7342 | EMA Qini: 0.7342 | Best EMA: 0.7342 ⭐ NEW BEST EMA
Epoch 2/150 | Base Loss: 262.4183 | Uplift Loss: 418.606232 | Total Loss: 681.0245 | Val Loss: 498.7781 | Val Qini: 0.7260 | EMA Qini: 0.7322 | Best EMA: 0.7342 (patience: 0/20)
Epoch 3/150 | Base Loss: 289.3818 | Uplift Loss: 341.326202 | Total Loss: 630.7080 | Val Loss: 498.6027 | Val Qini: 0.6941 | EMA Qini: 0.7227 | Best EMA: 0.7342 (patience: 0/20)
Epoch 4/150 | Base Loss: 656.5126 | Uplift Loss: 35